# Automated Marine Debris Detection using Deep Learning

## 1. Introduction and Problem Statement

## 2. Data Sources

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

### 3.2 Class Distribution

### 3.3 Image and Annotation Visualisation

### 3.4 Preprocessing and Data Splitting

## 4. Models and Methods

### 4.1 Faster R-CNN

### 4.2 Mask R-CNN

In [1]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor
)
import torch


def create_model(num_classes):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )
    return model


## 5. Experimental Setup

### 5.1 Training Configuration

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms.functional import pil_to_tensor
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
from torchvision.ops.boxes import masks_to_boxes
from torchvision.io import read_image
from pycocotools.coco import COCO

class MarineDataset(Dataset):
    def __init__(self, image_dir, annotation_file, category_ids, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_file)
        self.image_ids = sorted(self.coco.getImgIds())
        self.category_ids = category_ids
        self.transforms = transforms
    
        self.cat_to_label = {
            cat_id: i + 1
            for i, cat_id in enumerate(category_ids)
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs(image_id)[0]

        image_path = os.path.join(self.image_dir, info["file_name"])
        image = read_image(image_path)

        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        annotations = self.coco.loadAnns(ann_ids)

        boxes, labels, masks = [], [], []

        for ann in annotations:

            mask = torch.as_tensor(self.coco.annToMask(ann), dtype=torch.uint8)

            box = masks_to_boxes(mask.unsqueeze(0))[0]

            boxes.append(box)
            labels.append(self.cat_to_label[ann["category_id"]])
            masks.append(mask)

        num_objs = len(labels)
        boxes = torch.stack(boxes).float()
        boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(image))
        labels = torch.as_tensor(labels, dtype=torch.int64)
        masks = torch.stack(masks)
        masks = tv_tensors.Mask(masks)
        image_id = int(image_id)
        area = ((boxes[:, 3] - boxes[:, 1]) *(boxes[:, 2] - boxes[:, 0]))
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        image = tv_tensors.Image(image)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


In [ ]:
image, target = dataset[0]

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)

for key, value in target.items():
    if isinstance(value, torch.Tensor):
        print(key, value.shape, value.dtype)
    else:
        print(key, value)

n = len(target["labels"])

assert image.shape[0] == 3
assert image.dtype == torch.float32
assert target["boxes"].shape == (n, 4)
assert target["masks"].shape == (
    n, image.shape[-2], image.shape[-1]
)
assert target["labels"].dtype == torch.int64
assert target["area"].shape == (n,)
assert target["iscrowd"].shape == (n,)

print("Dataset format OK!")

In [ ]:

from help_funciton.engine import train_one_epoch, evaluate
import utils
# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

# our dataset has two classes only - background and person
image_dir = "/Users/bamboo/Desktop/comp9444/group project/comp9444-Automated-Marine-Debris-Detection-using-Deep-Learning/instance_version/train"
annotation_file = "/Users/bamboo/Desktop/comp9444/group project/comp9444-Automated-Marine-Debris-Detection-using-Deep-Learning/instance_version/instances_train_trashcan.json"
category_id = COCO(annotation_file).getCatIds()
num_classes = len(category_id) + 1
# use our dataset and defined transformations
dataset = MarineDataset(image_dir, annotation_file, category_id)
image_dir_val = "/Users/bamboo/Desktop/comp9444/group project/comp9444-Automated-Marine-Debris-Detection-using-Deep-Learning/instance_version/val"
annotation_file_val = "/Users/bamboo/Desktop/comp9444/group project/comp9444-Automated-Marine-Debris-Detection-using-Deep-Learning/instance_version/instances_val_trashcan.json"
category_id_val = COCO(annotation_file_val).getCatIds()
num_classes_val = len(category_id_val) + 1
dataset_val = MarineDataset(image_dir_val, annotation_file_val, category_id_val)

# # split the dataset in train and test set
# indices = torch.randperm(len(dataset)).tolist()
# dataset = torch.utils.data.Subset(dataset, indices[:-50])
# dataset_val = torch.utils.data.Subset(dataset_val, indices[-50:])

# define training and validation data loaders
def collate_fn(batch):
    return tuple(zip(*batch))
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

# get the model using our helper function
model = create_model(num_classes)

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_val, device=device)


loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
loading annotations into memory...
Done (t=0.31s)
creating index...
index created!
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /Users/bamboo/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth


100%|██████████| 177M/177M [00:17<00:00, 10.4MB/s] 
/Users/bamboo/Desktop/comp9444/group project/comp9444-Automated-Marine-Debris-Detection-using-Deep-Learning/help_funciton/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [   0/3033]  eta: 19 days, 20:41:31  lr: 0.000010  loss: 4.3387 (4.3387)  loss_classifier: 3.3777 (3.3777)  loss_box_reg: 0.1327 (0.1327)  loss_mask: 0.7699 (0.7699)  loss_objectness: 0.0508 (0.0508)  loss_rpn_box_reg: 0.0077 (0.0077)  time: 565.8066  data: 0.0627


### 5.2 Evaluation Metrics

## 6. Results

### 6.1 Quantitative Results

### 6.2 Qualitative Results

### 6.3 Model Comparison

## 7. Discussion

### 7.1 Error Analysis and Limitations

## 8. Conclusion and Future Work

## References